# Tennessee 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Tennessee, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals).

**Output**: A single CSV where each row is a county and columns include:

- Primary per-candidate vote counts (prefixed with `pri_`)
- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_primary_total`, `dem_primary_total`, `rep_general_total`, `dem_general_total`, `ind_general_total`

**Last Updated**: 2025/10/22

## 0. Library Import

In [1]:
import re 
import pandas as pd
import numpy as numpy
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

In [4]:
# TN 2008 dataset path
PRIMARY_PATH = r"../../data/raw/2008/TN/20080205__tn__primary__president__precinct.csv"
GENERAL_PATH = r"../../data/raw/2008/TN/20081104__tn__general__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/TN/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### a. Primary Election Dataset

In [5]:
# Load primary data
primary_df = pd.read_csv(PRIMARY_PATH)
primary_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,ANDERSON,Andersonville,Presidential Preference,NaN,Democratic,Joe Biden,2.0
1,ANDERSON,Andersonville,Presidential Preference,NaN,Democratic,Hillary Clinton,163.0
2,ANDERSON,Andersonville,Presidential Preference,NaN,Democratic,John Edwards,25.0
3,ANDERSON,Andersonville,Presidential Preference,NaN,Democratic,Mike Gravel,1.0
4,ANDERSON,Andersonville,Presidential Preference,NaN,Democratic,Barack Obama,48.0
5,ANDERSON,Andersonville,Presidential Preference,NaN,Democratic,Bill Richardson,2.0
6,ANDERSON,Andersonville,Presidential Preference,NaN,Republican,Rudy Giuliani,3.0
7,ANDERSON,Andersonville,Presidential Preference,NaN,Republican,Mike Huckabee,133.0
8,ANDERSON,Andersonville,Presidential Preference,NaN,Republican,Alan Keyes,2.0
9,ANDERSON,Andersonville,Presidential Preference,NaN,Republican,John McCain,93.0


In [6]:
# Primary data shape
primary_df.shape

(25087, 7)

In [7]:
# Number of missing values in each column
primary_df.isna().sum()

county           0
precinct         0
office           0
district     25087
party            0
candidate        0
votes            0
dtype: int64

In [10]:
# Number of unique values in each column
primary_df.nunique(dropna=True)

county         95
precinct     2221
office          1
district        0
party           2
candidate      18
votes         605
dtype: int64

In [11]:
# Drop the "office" column since it only has a single value
# Also, drop the district column since it's all missing values
primary_df = primary_df.drop(columns=["office", "district"]).reset_index(drop=True)
primary_df.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,ANDERSON,Andersonville,Democratic,Joe Biden,2.0
1,ANDERSON,Andersonville,Democratic,Hillary Clinton,163.0
2,ANDERSON,Andersonville,Democratic,John Edwards,25.0
3,ANDERSON,Andersonville,Democratic,Mike Gravel,1.0
4,ANDERSON,Andersonville,Democratic,Barack Obama,48.0
5,ANDERSON,Andersonville,Democratic,Bill Richardson,2.0
6,ANDERSON,Andersonville,Republican,Rudy Giuliani,3.0
7,ANDERSON,Andersonville,Republican,Mike Huckabee,133.0
8,ANDERSON,Andersonville,Republican,Alan Keyes,2.0
9,ANDERSON,Andersonville,Republican,John McCain,93.0


Given that the general election data is given on precinct-level, we will compute county-level vote totals by grouping on precinct and summing the precinct votes.

In [12]:
# Calculate county-level votes
primary_df = (
    primary_df.groupby(["county", "candidate", "party"], as_index=False)["votes"].sum()
)

primary_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Alan Keyes,Republican,19.0
1,ANDERSON,Barack Obama,Democratic,2559.0
2,ANDERSON,Bill Richardson,Democratic,30.0
3,ANDERSON,Chris Dodd,Democratic,7.0
4,ANDERSON,Dennis Kucinich,Democratic,15.0
5,ANDERSON,Duncan Hunter,Republican,7.0
6,ANDERSON,Fred Thompson,Republican,187.0
7,ANDERSON,Hillary Clinton,Democratic,4887.0
8,ANDERSON,Joe Biden,Democratic,19.0
9,ANDERSON,John Edwards,Democratic,445.0


In [13]:
# Different candidates in primary_df
primary_df["candidate"].value_counts()

candidate
Uncommitted        181
Barack Obama        95
Rudy Giuliani       95
Ron Paul            95
Mitt Romney         95
Mike Huckabee       95
John McCain         95
John Edwards        95
Hillary Clinton     95
Fred Thompson       95
Joe Biden           93
Bill Richardson     93
Duncan Hunter       89
Dennis Kucinich     88
Chris Dodd          84
Mike Gravel         81
Alan Keyes          80
Tom Tancredo        54
Name: count, dtype: int64

In [14]:
# List out all the parties in the general election data
primary_df["party"].value_counts()

party
Republican    880
Democratic    818
Name: count, dtype: int64

In [15]:
# Data type of each column in primary_df
primary_df.dtypes

county        object
candidate     object
party         object
votes        float64
dtype: object

Note that, for `votes`, we expect the data type to be integer ("int64"). Thus, we will coerce it not to avoid any future error.

In [18]:
# Coerce "votes" to have integer type
primary_df["votes"] = pd.to_numeric(primary_df["votes"], errors="coerce").astype("int64")

# Check the data type again
primary_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [19]:
# Final look at the cleaned primary_df
primary_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Alan Keyes,Republican,19
1,ANDERSON,Barack Obama,Democratic,2559
2,ANDERSON,Bill Richardson,Democratic,30
3,ANDERSON,Chris Dodd,Democratic,7
4,ANDERSON,Dennis Kucinich,Democratic,15
5,ANDERSON,Duncan Hunter,Republican,7
6,ANDERSON,Fred Thompson,Republican,187
7,ANDERSON,Hillary Clinton,Democratic,4887
8,ANDERSON,Joe Biden,Democratic,19
9,ANDERSON,John Edwards,Democratic,445


In [20]:
# Shape after preprocessing
primary_df.shape

(1698, 4)

### b. General Election Dataset

In [21]:
# Load general data
general_df = pd.read_csv(GENERAL_PATH)
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,ANDERSON,Andersonville,President,NaN,D,Barack Obama,290.0
1,ANDERSON,Andersonville,President,NaN,R,John McCain,1004.0
2,ANDERSON,Andersonville,President,NaN,I,Chuck Baldwin,3.0
3,ANDERSON,Andersonville,President,NaN,I,Bob Barr,6.0
4,ANDERSON,Andersonville,President,NaN,I,Charles Jay,2.0
5,ANDERSON,Andersonville,President,NaN,I,Brian Moore,1.0
6,ANDERSON,Andersonville,President,NaN,I,Ralph Nader,7.0
7,ANDERSON,Andersonville,United States Senate,NaN,D,Robert DTuke,221.0
8,ANDERSON,Andersonville,United States Senate,NaN,R,Lamar Alexander,989.0
9,ANDERSON,Andersonville,United States Senate,NaN,I,Edward LBuck,13.0


In [22]:
# Different values in 'office' column
general_df["office"].value_counts()

office
United States Senate                      16410
President                                 13226
U.S. House of Representatives District     7143
State House District                       3895
State Senate District                      2080
Name: count, dtype: int64

In [23]:
# Only keep rows where 'office' is 'President'
general_df = general_df[general_df["office"] == "President"]
general_df.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,ANDERSON,Andersonville,President,NaN,D,Barack Obama,290.0
1,ANDERSON,Andersonville,President,NaN,R,John McCain,1004.0
2,ANDERSON,Andersonville,President,NaN,I,Chuck Baldwin,3.0
3,ANDERSON,Andersonville,President,NaN,I,Bob Barr,6.0
4,ANDERSON,Andersonville,President,NaN,I,Charles Jay,2.0
5,ANDERSON,Andersonville,President,NaN,I,Brian Moore,1.0
6,ANDERSON,Andersonville,President,NaN,I,Ralph Nader,7.0
20,ANDERSON,Briceville,President,NaN,D,Barack Obama,120.0
21,ANDERSON,Briceville,President,NaN,R,John McCain,198.0
22,ANDERSON,Briceville,President,NaN,I,Chuck Baldwin,2.0


In [24]:
# General data shape when only considering President/VicePresident
general_df.shape

(13226, 7)

In [25]:
# Number of missing values in each column
general_df.isna().sum()

county           0
precinct         0
office           0
district     13226
party            0
candidate        0
votes            0
dtype: int64

In [27]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the "district" column since it's all missing values
general_df = general_df.drop(columns=["office", "district"]).reset_index(drop=True)
general_df.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,ANDERSON,Andersonville,D,Barack Obama,290.0
1,ANDERSON,Andersonville,R,John McCain,1004.0
2,ANDERSON,Andersonville,I,Chuck Baldwin,3.0
3,ANDERSON,Andersonville,I,Bob Barr,6.0
4,ANDERSON,Andersonville,I,Charles Jay,2.0
5,ANDERSON,Andersonville,I,Brian Moore,1.0
6,ANDERSON,Andersonville,I,Ralph Nader,7.0
7,ANDERSON,Briceville,D,Barack Obama,120.0
8,ANDERSON,Briceville,R,John McCain,198.0
9,ANDERSON,Briceville,I,Chuck Baldwin,2.0


Now, we will groupby `county` to get county-level vote counts.

In [32]:
# Calculate county-level votes
general_df = (
    general_df.groupby(["county", "candidate", "party"], as_index=False)["votes"].sum()
)

general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Barack Obama,D,11396
1,ANDERSON,Bob Barr,I,146
2,ANDERSON,Brian Moore,I,11
3,ANDERSON,Charles Jay,I,9
4,ANDERSON,Chuck Baldwin,I,134
5,ANDERSON,Cynthia McKinney,I,24
6,ANDERSON,John McCain,R,19675
7,ANDERSON,Ralph Nader,I,175
8,BEDFORD,Barack Obama,D,5027
9,BEDFORD,Bob Barr,I,51


In [33]:
# Candidates in general_df
general_df["candidate"].value_counts()

candidate
Barack Obama        95
Bob Barr            95
Chuck Baldwin       95
Cynthia McKinney    95
John McCain         95
Ralph Nader         95
Brian Moore         92
Charles Jay         92
Name: count, dtype: int64

In [34]:
# List out all the parties in the general election data
general_df["party"].value_counts()

party
I    564
D     95
R     95
Name: count, dtype: int64

In [35]:
# Data type of each column in general_df
general_df.dtypes

county       object
candidate    object
party        object
votes         int64
dtype: object

In [36]:
# Final look at the cleaned general_df
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,ANDERSON,Barack Obama,D,11396
1,ANDERSON,Bob Barr,I,146
2,ANDERSON,Brian Moore,I,11
3,ANDERSON,Charles Jay,I,9
4,ANDERSON,Chuck Baldwin,I,134
5,ANDERSON,Cynthia McKinney,I,24
6,ANDERSON,John McCain,R,19675
7,ANDERSON,Ralph Nader,I,175
8,BEDFORD,Barack Obama,D,5027
9,BEDFORD,Bob Barr,I,51


In [37]:
# Shape after preprocessing
general_df.shape

(754, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [46]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "Democratic" : "dem", "D" : "dem",                 
                "Republican" : "rep", "R" : "rep",
                "I"          : "ind"
               })
           .fillna(s.str.strip().str.lower()))      # For all others, they're all three-letter abbr, just need to lowercase

In [47]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [48]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [49]:
# Primary dataframe pivot
primary_pivot = pivot_wide(primary_df, prefix="pri")
primary_pivot.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNCOMMITTED,pri_rep_GIULIANI,pri_rep_HUCKABEE,pri_rep_HUNTER,pri_rep_KEYES,pri_rep_MCCAIN,pri_rep_PAUL,pri_rep_ROMNEY,pri_rep_TANCREDO,pri_rep_THOMPSON,pri_rep_UNCOMMITTED
0,ANDERSON,19,4887,7,445,11,15,2559,30,28,98,2369,7,19,2314,499,1547,3,187,10
1,BEDFORD,11,2934,4,204,2,3,900,8,26,17,1230,5,7,885,143,798,1,89,6
2,BENTON,16,2200,5,421,10,1,368,17,60,5,307,0,1,301,59,120,0,36,0
3,BLEDSOE,10,1399,0,132,1,1,195,6,25,8,577,4,3,387,51,162,0,37,2
4,BLOUNT,20,5719,6,397,4,18,3090,17,34,157,5144,29,31,4920,1242,3010,0,382,48
5,BRADLEY,12,4139,5,377,7,1,1625,10,10,88,5887,7,8,3102,583,1623,1,257,8
6,CAMPBELL,17,2854,4,122,2,2,326,7,17,23,1007,6,2,833,73,480,1,56,6
7,CANNON,6,1572,1,119,2,0,264,1,15,4,379,4,0,362,93,281,0,38,1
8,CARROLL,9,1982,6,157,1,5,709,1,5,17,767,2,3,765,93,324,1,65,4
9,CARTER,4,2367,4,189,2,3,747,7,12,70,3178,5,10,2421,405,1192,4,261,36


Note that there are two columns with uncommitted candidate that still had votes (`pri_dem_UNCOMMITTED` and `pri_rep_UNCOMMITTED`). We will keep this for total counting purposes and drop it at the end.

In [50]:
# Primary dataframe shape after pivot
primary_pivot.shape

(95, 20)

In [51]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_dem_OBAMA,gen_ind_BALDWIN,gen_ind_BARR,gen_ind_JAY,gen_ind_MCKINNEY,gen_ind_MOORE,gen_ind_NADER,gen_rep_MCCAIN
0,ANDERSON,11396,134,146,9,24,11,175,19675
1,BEDFORD,5027,76,51,4,17,11,104,10217
2,BENTON,2645,38,21,2,16,5,56,3696
3,BLEDSOE,1517,27,17,6,10,9,32,3166
4,BLOUNT,15253,245,239,23,41,13,260,35571
5,BRADLEY,9357,134,127,14,45,24,157,28333
6,CAMPBELL,3867,61,34,5,18,9,99,8535
7,CANNON,2011,29,27,2,11,5,50,3322
8,CARROLL,3980,55,37,7,19,7,86,7455
9,CARTER,5587,77,80,10,31,21,111,15852


In [52]:
# General dataframe shape after pivot
general_pivot.shape

(95, 9)

## 4. Merge Dataframes

Before merging, we verify that county names match across primary and general:

In [53]:
# Check if county names match between primary_df and general_df
primary_counties = set(primary_pivot["county"].unique())
general_counties = set(general_pivot["county"].unique())
common_counties = primary_counties.intersection(general_counties)
print(f"Number of common counties: {len(common_counties)} out of {len(primary_counties)}")

Number of common counties: 95 out of 95


Great. Since we know that all counties name are matched, we don't need to perform further data preprocessing to match the county names. Thus, we can now merge them:

In [54]:
# Merge primary and general dataframes on 'county'
merged_df = primary_pivot.merge(general_pivot, on="county", how="inner").fillna(0)    # There should be no missing values to fill with 0
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNCOMMITTED,...,pri_rep_THOMPSON,pri_rep_UNCOMMITTED,gen_dem_OBAMA,gen_ind_BALDWIN,gen_ind_BARR,gen_ind_JAY,gen_ind_MCKINNEY,gen_ind_MOORE,gen_ind_NADER,gen_rep_MCCAIN
0,ANDERSON,19,4887,7,445,11,15,2559,30,28,...,187,10,11396,134,146,9,24,11,175,19675
1,BEDFORD,11,2934,4,204,2,3,900,8,26,...,89,6,5027,76,51,4,17,11,104,10217
2,BENTON,16,2200,5,421,10,1,368,17,60,...,36,0,2645,38,21,2,16,5,56,3696
3,BLEDSOE,10,1399,0,132,1,1,195,6,25,...,37,2,1517,27,17,6,10,9,32,3166
4,BLOUNT,20,5719,6,397,4,18,3090,17,34,...,382,48,15253,245,239,23,41,13,260,35571
5,BRADLEY,12,4139,5,377,7,1,1625,10,10,...,257,8,9357,134,127,14,45,24,157,28333
6,CAMPBELL,17,2854,4,122,2,2,326,7,17,...,56,6,3867,61,34,5,18,9,99,8535
7,CANNON,6,1572,1,119,2,0,264,1,15,...,38,1,2011,29,27,2,11,5,50,3322
8,CARROLL,9,1982,6,157,1,5,709,1,5,...,65,4,3980,55,37,7,19,7,86,7455
9,CARTER,4,2367,4,189,2,3,747,7,12,...,261,36,5587,77,80,10,31,21,111,15852


In [55]:
# Statistics check on merged dataframe 
merged_df.describe()

,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_dem_UNCOMMITTED,pri_rep_GIULIANI,...,pri_rep_THOMPSON,pri_rep_UNCOMMITTED,gen_dem_OBAMA,gen_ind_BALDWIN,gen_ind_BARR,gen_ind_JAY,gen_ind_MCKINNEY,gen_ind_MOORE,gen_ind_NADER,gen_rep_MCCAIN
count,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,...,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000
mean,16.115789,3539.421053,5.536842,292.842105,4.852632,10.221053,2661.831579,12.400000,33.242105,54.305263,...,171.189474,19.263158,11446.705263,86.221053,89.968421,10.642105,26.305263,13.957895,121.684211,15570.294737
std,29.220316,4870.704070,7.723700,347.463371,8.051653,23.282803,8991.878051,18.365903,51.389360,115.232058,...,266.594093,50.907344,31685.221611,118.123644,158.422036,16.638788,40.578286,20.453328,173.615818,23770.881993
min,0.000000,240.000000,0.000000,28.000000,0.000000,0.000000,56.000000,0.000000,0.000000,2.000000,...,7.000000,0.000000,604.000000,3.000000,6.000000,0.000000,1.000000,0.000000,15.000000,1175.000000
25%,5.000000,1310.000000,1.000000,96.500000,1.000000,2.000000,239.500000,3.500000,6.000000,10.000000,...,38.000000,2.000000,2035.500000,27.000000,22.500000,4.000000,10.000000,5.000000,38.000000,4176.000000
50%,9.000000,2367.000000,4.000000,204.000000,3.000000,4.000000,654.000000,8.000000,17.000000,18.000000,...,77.000000,6.000000,4320.000000,46.000000,38.000000,6.000000,16.000000,9.000000,70.000000,7669.000000
75%,17.000000,3542.000000,7.000000,341.000000,4.500000,9.000000,1339.000000,13.000000,32.500000,54.500000,...,216.000000,21.500000,7178.000000,89.500000,82.500000,10.500000,26.500000,15.000000,120.000000,15736.500000
max,247.000000,32338.000000,55.000000,2146.000000,46.000000,154.000000,68551.000000,113.000000,265.000000,985.000000,...,2118.000000,465.000000,256297.000000,619.000000,963.000000,103.000000,279.000000,155.000000,1164.000000,145458.000000


Now, we will add party totals columns: 

- Primary totals:
    * `rep_primary_total` = sum of all `pri_rep_*` columns
    * `dem_primary_total` = sum of all `pri_dem_*` columns

- General totals:
    * `rep_general_total` = sum of all `gen_rep_*` columns
    * `dem_general_total` = sum of all `gen_dem_*` columns
    * `ind_general_total` = sum of all `gen_ind_*` columns

In [56]:
# Add party totals for primary election
rep_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_rep_")]
dem_primary_cols   = [c for c in merged_df.columns if c.startswith("pri_dem_")]

merged_df["rep_primary_total"] = merged_df[rep_primary_cols].sum(axis=1) if rep_primary_cols else 0
merged_df["dem_primary_total"] = merged_df[dem_primary_cols].sum(axis=1) if dem_primary_cols else 0

Now, we have calculated the total vote for each party. Thus, we can drop the two UNCOMMITTED columns.

In [57]:
# Drop UNCOMMITTED columns for primary election
merged_df = merged_df.drop(columns=["pri_rep_UNCOMMITTED", "pri_dem_UNCOMMITTED"])

# Snippet at the merged dataframe with primary totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,...,gen_dem_OBAMA,gen_ind_BALDWIN,gen_ind_BARR,gen_ind_JAY,gen_ind_MCKINNEY,gen_ind_MOORE,gen_ind_NADER,gen_rep_MCCAIN,rep_primary_total,dem_primary_total
0,ANDERSON,19,4887,7,445,11,15,2559,30,98,...,11396,134,146,9,24,11,175,19675,7053,8001
1,BEDFORD,11,2934,4,204,2,3,900,8,17,...,5027,76,51,4,17,11,104,10217,3181,4092
2,BENTON,16,2200,5,421,10,1,368,17,5,...,2645,38,21,2,16,5,56,3696,829,3098
3,BLEDSOE,10,1399,0,132,1,1,195,6,8,...,1517,27,17,6,10,9,32,3166,1231,1769
4,BLOUNT,20,5719,6,397,4,18,3090,17,157,...,15253,245,239,23,41,13,260,35571,14963,9305
5,BRADLEY,12,4139,5,377,7,1,1625,10,88,...,9357,134,127,14,45,24,157,28333,11564,6186
6,CAMPBELL,17,2854,4,122,2,2,326,7,23,...,3867,61,34,5,18,9,99,8535,2487,3351
7,CANNON,6,1572,1,119,2,0,264,1,4,...,2011,29,27,2,11,5,50,3322,1162,1980
8,CARROLL,9,1982,6,157,1,5,709,1,17,...,3980,55,37,7,19,7,86,7455,2041,2875
9,CARTER,4,2367,4,189,2,3,747,7,70,...,5587,77,80,10,31,21,111,15852,7582,3335


In [58]:
# Add party totals for general election
rep_general_cols   = [c for c in merged_df.columns if c.startswith("gen_rep_")]
dem_general_cols   = [c for c in merged_df.columns if c.startswith("gen_dem_")]
ind_general_cols   = [c for c in merged_df.columns if c.startswith("gen_ind_")]

merged_df["rep_general_total"] = merged_df[rep_general_cols].sum(axis=1) if rep_general_cols else 0
merged_df["dem_general_total"] = merged_df[dem_general_cols].sum(axis=1) if dem_general_cols else 0
merged_df["ind_general_total"] = merged_df[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [59]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned dataframe:")
merged_df.columns

Final columns in the cleaned dataframe:


Index(['county', 'pri_dem_BIDEN', 'pri_dem_CLINTON', 'pri_dem_DODD',
       'pri_dem_EDWARDS', 'pri_dem_GRAVEL', 'pri_dem_KUCINICH',
       'pri_dem_OBAMA', 'pri_dem_RICHARDSON', 'pri_rep_GIULIANI',
       'pri_rep_HUCKABEE', 'pri_rep_HUNTER', 'pri_rep_KEYES', 'pri_rep_MCCAIN',
       'pri_rep_PAUL', 'pri_rep_ROMNEY', 'pri_rep_TANCREDO',
       'pri_rep_THOMPSON', 'gen_dem_OBAMA', 'gen_ind_BALDWIN', 'gen_ind_BARR',
       'gen_ind_JAY', 'gen_ind_MCKINNEY', 'gen_ind_MOORE', 'gen_ind_NADER',
       'gen_rep_MCCAIN', 'rep_primary_total', 'dem_primary_total',
       'rep_general_total', 'dem_general_total', 'ind_general_total'],
      dtype='object')

In [60]:
# Preview merged dataframe with totals
merged_df.head(DISPLAY_ROWS)

,county,pri_dem_BIDEN,pri_dem_CLINTON,pri_dem_DODD,pri_dem_EDWARDS,pri_dem_GRAVEL,pri_dem_KUCINICH,pri_dem_OBAMA,pri_dem_RICHARDSON,pri_rep_GIULIANI,...,gen_ind_JAY,gen_ind_MCKINNEY,gen_ind_MOORE,gen_ind_NADER,gen_rep_MCCAIN,rep_primary_total,dem_primary_total,rep_general_total,dem_general_total,ind_general_total
0,ANDERSON,19,4887,7,445,11,15,2559,30,98,...,9,24,11,175,19675,7053,8001,19675,11396,499
1,BEDFORD,11,2934,4,204,2,3,900,8,17,...,4,17,11,104,10217,3181,4092,10217,5027,263
2,BENTON,16,2200,5,421,10,1,368,17,5,...,2,16,5,56,3696,829,3098,3696,2645,138
3,BLEDSOE,10,1399,0,132,1,1,195,6,8,...,6,10,9,32,3166,1231,1769,3166,1517,101
4,BLOUNT,20,5719,6,397,4,18,3090,17,157,...,23,41,13,260,35571,14963,9305,35571,15253,821
5,BRADLEY,12,4139,5,377,7,1,1625,10,88,...,14,45,24,157,28333,11564,6186,28333,9357,501
6,CAMPBELL,17,2854,4,122,2,2,326,7,23,...,5,18,9,99,8535,2487,3351,8535,3867,226
7,CANNON,6,1572,1,119,2,0,264,1,4,...,2,11,5,50,3322,1162,1980,3322,2011,124
8,CARROLL,9,1982,6,157,1,5,709,1,17,...,7,19,7,86,7455,2041,2875,7455,3980,211
9,CARTER,4,2367,4,189,2,3,747,7,70,...,10,31,21,111,15852,7582,3335,15852,5587,330


Now, we save the cleaned dataframe into the processed directory.

In [61]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(OUTPUT_PATH + "TN.csv", index=False)